# 📊 Fase 6 — Backtesting MTM V4

Análisis de rendimiento del Score Log V4.

> **Cuándo ejecutar:** Viernes por la tarde.
> **Requisito:** Al menos 1 semana de datos en `📊 Score Log V4`.
> **Tiempo estimado:** 3-5 minutos.
> **Versión:** Detecta automáticamente si tu Score Log tiene 8 o 17 columnas.

**Preguntas que responde:**
- ¿Las de **Alta Confianza** (Score ≥ 4.5) tuvieron mejor win rate?
- ¿Qué **sector** tuvo mejor performance? (si hay datos)
- ¿Los setups con **R/R ≥ 2.0** funcionaron mejor? (si hay datos)
- ¿Cuál fue el **drawdown** máximo?
- ¿Cuánto hubiera crecido un **portafolio** siguiendo el sistema?
- ¿Y si solo operaba las **3 mejores** del Radar?

## Paso 0: Instalar librerías

In [ ]:
!pip install yfinance gspread pandas numpy matplotlib -q
print('✅ Librerías instaladas')

## Paso 1: Conectar con Google Sheets

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_ID = 'TU_SPREADSHEET_ID_AQUI'  # ← Pegá el ID de tu Tracker V4
ss = gc.open_by_key(SPREADSHEET_ID)
print('✅ Conectado a:', ss.title)
print('Hojas disponibles:', [ws.title for ws in ss.worksheets()])

## Paso 2: Leer Score Log V4 (detecta automáticamente la versión)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

SHEET_LOG = '📊 Score Log V4'

ws = ss.worksheet(SHEET_LOG)
data = ws.get_all_values()

# El header está en fila 2 (fila 1 es título)
headers = data[1]
rows = data[2:]

# Detectar versión (8 cols = vieja, 17 cols = nueva)
VERSION_V4 = len(headers) >= 17

df = pd.DataFrame(rows, columns=headers)
print(f'✅ Score Log leído: {len(df)} registros')
print(f'Versión detectada: {"V4 COMPLETA (17 cols)" if VERSION_V4 else "V4 BÁSICA (8 cols)"}')
print(f'Columnas: {headers}')

display(df.head())

## Paso 3: Limpiar y preparar datos (adapta según versión)

In [ ]:
# Reemplazar comas por puntos en TODO el dataframe primero (formato español)
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '.')

# Convertir fechas
df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')

# Convertir numéricas base
df['PRECIO'] = pd.to_numeric(df['PRECIO'], errors='coerce')
df['SCORE V4'] = pd.to_numeric(df['SCORE V4'], errors='coerce')

# Columnas opcionales
col_numericas = {
    'ENTRADA': 'ENTRADA', 'STOP': 'STOP', 'TARGET': 'TARGET',
    'R/R': 'R/R', 'PERF W': 'PERF W', 'PERF M': 'PERF M'
}
for col_excel, col_df in col_numericas.items():
    if col_excel in df.columns:
        df[col_df] = pd.to_numeric(df[col_excel], errors='coerce')
    else:
        df[col_df] = np.nan

# Columnas de texto opcionales
col_texto = ['SECTOR', 'ATR/LOW', 'SCTR', 'FUENTE', 'ESTADO']
for col in col_texto:
    if col not in df.columns:
        df[col] = ''

# Nivel de Confianza
def nivel_confianza(score):
    if score >= 4.5: return 'Alta'
    elif score >= 3.0: return 'Media'
    elif score >= 2.0: return 'Base'
    else: return 'Baja'

df['Nivel'] = df['SCORE V4'].apply(nivel_confianza)

# R/R Categorizado
def categoria_rr(rr):
    if pd.isna(rr): return 'Sin setup'
    if rr >= 2.5: return 'Excelente (≥2.5x)'
    elif rr >= 2.0: return 'Bueno (2.0-2.5x)'
    elif rr >= 1.5: return 'Regular (1.5-2.0x)'
    else: return 'Débil (<1.5x)'

df['R/R Cat'] = df['R/R'].apply(categoria_rr)

# Eliminar registros sin precio o score
df_validos = df.dropna(subset=['PRECIO', 'SCORE V4']).copy()

print(f'✅ Registros válidos: {len(df_validos)} de {len(df)}')
print(f'Rango de fechas: {df_validos["FECHA"].min()} a {df_validos["FECHA"].max()}')
print(f'Niveles de confianza:')
print(df_validos['Nivel'].value_counts())

## Paso 4: Descargar precios de cierre (Yahoo Finance)

> ⚠️ Esto puede tardar 2-3 minutos.

In [ ]:
import yfinance as yf
import time

def obtener_precio_semana_siguiente(ticker, fecha):
    try:
        t = yf.Ticker(ticker)
        dias_dif = (datetime.now() - fecha).days
        if dias_dif < 7:
            hist = t.history(period='5d')
            if len(hist) > 0:
                return hist['Close'].iloc[-1]
            return None
        inicio = fecha.strftime('%Y-%m-%d')
        fin = (fecha + timedelta(days=14)).strftime('%Y-%m-%d')
        hist = t.history(start=inicio, end=fin)
        if len(hist) < 2: return None
        idx = min(5, len(hist) - 1)
        return hist['Close'].iloc[idx]
    except Exception as e:
        return None

print('📥 Descargando precios de Yahoo Finance...')
precios_cierre = {}
unicos = df_validos[['TICKER', 'FECHA']].drop_duplicates()

for i, row in unicos.iterrows():
    tk = row['TICKER']
    fecha = row['FECHA']
    key = f"{tk}_{fecha.strftime('%Y%m%d')}"
    precio = obtener_precio_semana_siguiente(tk, fecha)
    precios_cierre[key] = precio
    if (i + 1) % 10 == 0:
        print(f'  ...{i+1}/{len(unicos)} tickers descargados')
    time.sleep(0.3)

validos = len([v for v in precios_cierre.values() if v is not None])
print(f'✅ Precios descargados: {validos}/{len(unicos)}')

if validos == 0:
    print('⚠️ No se pudieron descargar precios.')

## Paso 5: Calcular retornos

In [ ]:
def calcular_retorno(row):
    key = f"{row['TICKER']}_{row['FECHA'].strftime('%Y%m%d')}"
    precio_cierre = precios_cierre.get(key)
    if precio_cierre is None or row['PRECIO'] == 0:
        return None
    return (precio_cierre - row['PRECIO']) / row['PRECIO']

df_validos['RETORNO_1S'] = df_validos.apply(calcular_retorno, axis=1)

def clasificar_ganadora(x):
    if x is None or pd.isna(x): return None
    return 'Sí' if x > 0 else 'No'

df_validos['GANADORA'] = df_validos['RETORNO_1S'].apply(clasificar_ganadora)

def simular_stop_target(row):
    retorno = row['RETORNO_1S']
    entrada = row.get('ENTRADA')
    stop = row.get('STOP')
    target = row.get('TARGET')
    if retorno is None or pd.isna(retorno) or pd.isna(entrada) or entrada == 0:
        return 'Sin setup'
    precio_final = row['PRECIO'] * (1 + retorno)
    if not pd.isna(stop) and precio_final <= stop: return 'Stop Hit'
    if not pd.isna(target) and precio_final >= target: return 'Target Hit'
    if retorno > 0: return 'Profit'
    else: return 'Loss'

df_validos['RESULTADO'] = df_validos.apply(simular_stop_target, axis=1)

print('✅ Retornos calculados')
print(f'Con datos de cierre: {len(df_validos.dropna(subset=["RETORNO_1S"]))} de {len(df_validos)}')
print('Resultados:')
print(df_validos['RESULTADO'].value_counts())

## Paso 6: Métricas de rendimiento — TODO EL RADAR vs MIS TRADES

In [ ]:
df_calc = df_validos.dropna(subset=['RETORNO_1S']).copy()

# ===== TODO EL RADAR =====
print('═' * 50)
print('📊 MÉTRICAS — TODO EL RADAR (todos los tickers)')
print('═' * 50)

total = len(df_calc)
ganadoras = len(df_calc[df_calc['RETORNO_1S'] > 0])
perdedoras = len(df_calc[df_calc['RETORNO_1S'] <= 0])
win_rate = ganadoras / total if total > 0 else 0

retorno_promedio = df_calc['RETORNO_1S'].mean()
retorno_std = df_calc['RETORNO_1S'].std()
sharpe = retorno_promedio / retorno_std if retorno_std > 0 else 0

cumulative = (1 + df_calc['RETORNO_1S']).cumprod()
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max
max_drawdown = drawdown.min()

print(f'Total trades: {total}')
print(f'Win Rate: {win_rate:.1%} ({ganadoras}/{total})')
print(f'Retorno promedio/trade: {retorno_promedio:.2%}')
print(f'Sharpe ratio: {sharpe:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Capital final (simulado $10k): ${10000*cumulative.iloc[-1]:,.2f}')

# ===== MIS TRADES (TRACKER? == SÍ) =====
print('\n')
print('═' * 50)
print('🎯 MÉTRICAS — SOLO MIS TRADES (Tracker con Track=SÍ)')
print('═' * 50)

df_mis_trades = df_calc[df_calc['TRACKER?'] == 'SÍ'].copy()

if len(df_mis_trades) > 0:
    total_mt = len(df_mis_trades)
    gan_mt = len(df_mis_trades[df_mis_trades['RETORNO_1S'] > 0])
    wr_mt = gan_mt / total_mt
    ret_mt = df_mis_trades['RETORNO_1S'].mean()
    cum_mt = (1 + df_mis_trades['RETORNO_1S']).cumprod()
    fin_mt = 10000 * cum_mt.iloc[-1]
    print(f'Total trades: {total_mt}')
    print(f'Win Rate: {wr_mt:.1%} ({gan_mt}/{total_mt})')
    print(f'Retorno promedio/trade: {ret_mt:+.2%}')
    print(f'Capital final (simulado $10k): ${fin_mt:,.2f}')
else:
    print('⚠️ No hay tickers con TRACKER? = SÍ.')
    print('   Agregá tickers al Tracker Diario antes de generar el Radar.')
    print('   El Score Log guarda TODOS los del Radar, pero solo los')
    print('   marcados en Tracker son "tus trades" reales.')

## Paso 7: Win Rate por Nivel de Confianza

In [ ]:
print('\n📊 WIN RATE POR NIVEL')
print('=' * 50)

for nivel in ['Alta', 'Media', 'Base']:
    subset = df_calc[df_calc['Nivel'] == nivel]
    if len(subset) == 0:
        print(f'{nivel}: Sin datos')
        continue
    wins = len(subset[subset['RETORNO_1S'] > 0])
    total_n = len(subset)
    wr = wins / total_n
    avg_ret = subset['RETORNO_1S'].mean()
    print(f'{nivel:6} | Win Rate: {wr:.1%} | Avg Return: {avg_ret:+.2%} | N: {total_n}')

nivel_stats = df_calc.groupby('Nivel').agg(
    Total=('RETORNO_1S', 'count'),
    Ganadoras=('GANADORA', lambda x: (x == 'Sí').sum()),
    Win_Rate=('GANADORA', lambda x: (x == 'Sí').mean()),
    Retorno_Prom=('RETORNO_1S', 'mean'),
    Mejor=('RETORNO_1S', 'max'),
    Peor=('RETORNO_1S', 'min')
).round(4)
display(nivel_stats)

## Paso 8: Análisis por Sector (solo si hay datos)

In [ ]:
if VERSION_V4 and 'SECTOR' in df_calc.columns and df_calc['SECTOR'].str.strip().any():
    print('\n📊 WIN RATE POR SECTOR')
    print('=' * 50)
    df_calc['SECTOR_LIMPIO'] = df_calc['SECTOR'].astype(str).str.strip().str.title()
    sector_stats = df_calc[df_calc['SECTOR_LIMPIO'] != ''].groupby('SECTOR_LIMPIO').agg(
        Total=('RETORNO_1S', 'count'),
        Ganadoras=('GANADORA', lambda x: (x == 'Sí').sum()),
        Win_Rate=('GANADORA', lambda x: (x == 'Sí').mean()),
        Retorno_Prom=('RETORNO_1S', 'mean')
    ).sort_values('Win_Rate', ascending=False)
    display(sector_stats)
else:
    print('⚠️ Análisis por Sector no disponible.')

## Paso 9: Análisis por R/R (Setup) — solo si hay datos

In [ ]:
if VERSION_V4 and 'R/R' in df_calc.columns and df_calc['R/R'].notna().any():
    print('\n📊 WIN RATE POR RIESGO/BENEFICIO')
    print('=' * 50)
    rr_stats = df_calc[df_calc['R/R Cat'] != 'Sin setup'].groupby('R/R Cat').agg(
        Total=('RETORNO_1S', 'count'),
        Ganadoras=('GANADORA', lambda x: (x == 'Sí').sum()),
        Win_Rate=('GANADORA', lambda x: (x == 'Sí').mean()),
        Retorno_Prom=('RETORNO_1S', 'mean')
    ).sort_values('Win_Rate', ascending=False)
    display(rr_stats)
else:
    print('⚠️ Análisis por R/R no disponible.')

## Paso 10: Curva de Equity (simulada)

In [ ]:
import matplotlib.pyplot as plt

capital_inicial = 10000
df_calc_sorted = df_calc.sort_values('FECHA')
df_calc_sorted['CAPITAL'] = capital_inicial * (1 + df_calc_sorted['RETORNO_1S']).cumprod()

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(df_calc_sorted.index, df_calc_sorted['CAPITAL'], color='#00C853', linewidth=2)
plt.axhline(y=capital_inicial, color='gray', linestyle='--', alpha=0.5)
plt.title('Curva de Equity', fontsize=14, fontweight='bold')
plt.xlabel('Número de Trade')
plt.ylabel('Capital ($)')
plt.grid(True, alpha=0.3)
plt.ticklabel_format(style='plain', axis='y')

plt.subplot(1, 2, 2)
colors = ['#FF1744' if r <= 0 else '#00C853' for r in df_calc['RETORNO_1S']]
plt.bar(range(len(df_calc)), df_calc['RETORNO_1S'] * 100, color=colors, alpha=0.7)
plt.axhline(y=0, color='black', linewidth=0.5)
plt.title('Retorno por Trade (%)', fontsize=14, fontweight='bold')
plt.xlabel('Trade #')
plt.ylabel('Retorno (%)')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

capital_final = df_calc_sorted['CAPITAL'].iloc[-1]
retorno_total = (capital_final - capital_inicial) / capital_inicial

print(f'\n💰 CAPITAL SIMULADO')
print(f'Capital inicial: ${capital_inicial:,.2f}')
print(f'Capital final: ${capital_final:,.2f}')
print(f'Retorno total: {retorno_total:.2%}')
print(f'Trades ejecutados: {len(df_calc)}')

## Paso 11: Top 10 Mejores y Peores

In [ ]:
cols_top = ['FECHA', 'TICKER', 'SCORE V4', 'Nivel']
if VERSION_V4:
    cols_top.extend(['R/R', 'ENTRADA'])

print('\n🏆 TOP 10 MEJORES SETUPS')
print('=' * 60)
mejores = df_calc.nlargest(10, 'RETORNO_1S')[cols_top + ['RETORNO_1S']]
mejores['RETORNO_1S'] = mejores['RETORNO_1S'].apply(lambda x: f'{x:+.2%}')
display(mejores)

print('\n💩 TOP 10 PEORES SETUPS')
print('=' * 60)
peores = df_calc.nsmallest(10, 'RETORNO_1S')[cols_top + ['RETORNO_1S']]
peores['RETORNO_1S'] = peores['RETORNO_1S'].apply(lambda x: f'{x:+.2%}')
display(peores)

## Paso 12: Guardar reporte en Google Sheets

In [ ]:
SHEET_BT = '📊 Backtesting Resumen'

try:
    ws_bt = ss.worksheet(SHEET_BT)
    ws_bt.clear()
except gspread.WorksheetNotFound:
    ws_bt = ss.add_worksheet(title=SHEET_BT, rows=50, cols=10)

resumen = [
    ['📊 BACKTESTING MTM V4 — Resumen Ejecutivo'],
    ['Generado el', str(datetime.now().strftime('%d/%m/%Y %H:%M'))],
    ['Período', f"{df_calc['FECHA'].min().strftime('%d/%m/%Y')} a {df_calc['FECHA'].max().strftime('%d/%m/%Y')}"],
    ['Versión Score Log', 'V4 COMPLETA (17 cols)' if VERSION_V4 else 'V4 BÁSICA (8 cols)'],
    [''],
    ['MÉTRICA', 'VALOR'],
    ['Total de trades (Radar)', total],
    ['Win Rate (Radar)', f'{win_rate:.1%}'],
    ['Retorno promedio/trade', f'{retorno_promedio:.2%}'],
    ['Sharpe Ratio (aprox)', f'{sharpe:.2f}'],
    ['Max Drawdown', f'{max_drawdown:.2%}'],
    ['Capital final (simulado)', f'${capital_final:,.2f}'],
    [''],
    ['WIN RATE POR NIVEL'],
    ['Nivel', 'Win Rate', 'Trades', 'Avg Return']
]

for nivel in ['Alta', 'Media', 'Base']:
    subset = df_calc[df_calc['Nivel'] == nivel]
    if len(subset) > 0:
        wr = (subset['GANADORA'] == 'Sí').mean()
        avg = subset['RETORNO_1S'].mean()
        resumen.append([nivel, f'{wr:.1%}', len(subset), f'{avg:+.2%}'])

ws_bt.update(range_name='A1', values=resumen)

print(f'✅ Backtesting guardado en hoja: {SHEET_BT}')
print('\n🎯 CONCLUSIONES:')
print('═' * 50)
if win_rate >= 0.55:
    print('✅ Win rate saludable (>55%).')
elif win_rate >= 0.45:
    print('⚠️ Win rate moderado (45-55%).')
else:
    print('❌ Win rate bajo (<45%). Considerar ajustar el sistema.')

if sharpe >= 1.5:
    print('✅ Sharpe ratio excelente (>1.5).')
elif sharpe >= 1.0:
    print('⚠️ Sharpe ratio aceptable (1.0-1.5).')
else:
    print('❌ Sharpe ratio bajo (<1.0).')

print('\n💡 Próximos pasos:')
print('   1. Si Alta Confianza tiene mejor win rate, priorizá esas.')
if VERSION_V4:
    print('   2. Si un sector domina, enfocate ahí.')
    print('   3. Si R/R >= 2.5 funciona mejor, exigí ese mínimo.')
else:
    print('   2. Generá el Radar con la versión completa para análisis por sector y R/R.')
print('   4. Acumulá más datos (3-6 meses) para conclusiones más sólidas.')

## Paso 13: Guardar copia de seguridad en Google Drive

In [ ]:
print('💾 Guardando copia de seguridad en Google Drive...')
from google.colab import drive
import os

# Montar Drive
drive.mount('/content/drive', force_remount=True)

# Crear directorio
dir_path = "/content/drive/My Drive/MTM_V4_Resultados"
os.makedirs(dir_path, exist_ok=True)

# Fecha para el nombre de archivo
fecha = datetime.now().strftime("%Y%m%d")
nombre_base = f"Backtesting_V4_{fecha}"

# ===== CONSTRUIR RESUMEN COMPLETO =====
resumen_txt = f"""
═══════════════════════════════════════════════════════════════════
BACKTESTING MTM V4 — {datetime.now().strftime('%d/%m/%Y %H:%M')}
═══════════════════════════════════════════════════════════════════

📅 PERÍODO ANALIZADO
   Desde: {df_calc['FECHA'].min().strftime('%d/%m/%Y')}
   Hasta: {df_calc['FECHA'].max().strftime('%d/%m/%Y')}
   Días transcurridos: {(df_calc['FECHA'].max() - df_calc['FECHA'].min()).days}
   Versión Score Log: {'V4 COMPLETA (17 cols)' if VERSION_V4 else 'V4 BÁSICA (8 cols)'}

═══════════════════════════════════════════════════════════════════
📊 MÉTRICAS GLOBALES — TODO EL RADAR
═══════════════════════════════════════════════════════════════════

Total de tickers evaluados: {total}
Con datos de cierre disponibles: {len(df_calc)}

Win Rate General: {win_rate:.1%} ({ganadoras} ganadoras / {perdedoras} perdedoras)
Retorno Promedio por trade: {retorno_promedio:+.2%}
Retorno Mediano por trade: {df_calc['RETORNO_1S'].median():+.2%}
Mejor trade: {df_calc['RETORNO_1S'].max():+.2%}
Peor trade: {df_calc['RETORNO_1S'].min():+.2%}
Desviación estándar: {retorno_std:.2%}
Sharpe Ratio (aprox): {sharpe:.2f}
Max Drawdown: {max_drawdown:.2%}

Capital inicial (simulado): $10,000.00
Capital final (simulado): ${capital_final:,.2f}
Retorno total del período: {((capital_final-10000)/10000):+.2%}

═══════════════════════════════════════════════════════════════════
🎯 MÉTRICAS — SOLO MIS TRADES (Tracker con Track=SÍ)
═══════════════════════════════════════════════════════════════════
"""

# Agregar sección de mis trades
if len(df_mis_trades) > 0:
    resumen_txt += f"""
Total de mis trades: {len(df_mis_trades)}
Win Rate: {wr_mt:.1%} ({gan_mt}/{len(df_mis_trades)})
Retorno promedio: {ret_mt:+.2%}
Capital final: ${fin_mt:,.2f}
"""
    for _, row in df_mis_trades.iterrows():
        resumen_txt += f"   • {row['TICKER']:8} | Score: {row['SCORE V4']:.2f} | Retorno: {row['RETORNO_1S']:+.2%}\n"
else:
    resumen_txt += """
⚠️ No hay tickers con TRACKER? = SÍ en este período.
   Recordá: Agregá tickers al Tracker Diario (col E = TRUE) antes de generar el Radar.
"""

resumen_txt += """
═══════════════════════════════════════════════════════════════════
📊 WIN RATE POR NIVEL DE CONFIANZA
═══════════════════════════════════════════════════════════════════

"""

for nivel in ['Alta', 'Media', 'Base', 'Baja']:
    subset = df_calc[df_calc['Nivel'] == nivel]
    if len(subset) > 0:
        wins = len(subset[subset['RETORNO_1S'] > 0])
        wr = wins / len(subset)
        avg_ret = subset['RETORNO_1S'].mean()
        resumen_txt += f"{nivel:8} | Win Rate: {wr:6.1%} | Avg Return: {avg_ret:+.2%} | Trades: {len(subset)}\n"
    else:
        resumen_txt += f"{nivel:8} | Sin datos en este período\n"

resumen_txt += """
═══════════════════════════════════════════════════════════════════
🏆 TOP 10 MEJORES SETUPS
═══════════════════════════════════════════════════════════════════

"""

top10 = df_calc.nlargest(10, 'RETORNO_1S')
for i, (_, row) in enumerate(top10.iterrows(), 1):
    rr_str = f"R/R: {row['R/R']:.1f}x" if pd.notna(row.get('R/R')) else "Sin setup"
    resumen_txt += f"{i:2}. {row['TICKER']:8} | Score: {row['SCORE V4']:.2f} | {rr_str:15} | Retorno: {row['RETORNO_1S']:+.2%}\n"

resumen_txt += """
═══════════════════════════════════════════════════════════════════
💩 TOP 10 PEORES SETUPS
═══════════════════════════════════════════════════════════════════

"""

worst10 = df_calc.nsmallest(10, 'RETORNO_1S')
for i, (_, row) in enumerate(worst10.iterrows(), 1):
    rr_str = f"R/R: {row['R/R']:.1f}x" if pd.notna(row.get('R/R')) else "Sin setup"
    resumen_txt += f"{i:2}. {row['TICKER']:8} | Score: {row['SCORE V4']:.2f} | {rr_str:15} | Retorno: {row['RETORNO_1S']:+.2%}\n"

resumen_txt += """
═══════════════════════════════════════════════════════════════════
💡 CONCLUSIONES Y RECOMENDACIONES
═══════════════════════════════════════════════════════════════════

"""

# Win rate general
if win_rate >= 0.55:
    resumen_txt += "✅ Win Rate: Saludable (>55%). El sistema está funcionando bien.\n"
elif win_rate >= 0.45:
    resumen_txt += "⚠️  Win Rate: Moderado (45-55%). Revisar filtros de entrada.\n"
else:
    resumen_txt += "❌ Win Rate: Bajo (<45%). Considerar ajustar umbrales o esperar más datos.\n"

# Sharpe
if sharpe >= 1.5:
    resumen_txt += "✅ Sharpe Ratio: Excelente (>1.5). Buena relación retorno/riesgo.\n"
elif sharpe >= 1.0:
    resumen_txt += "⚠️  Sharpe Ratio: Aceptable (1.0-1.5). Se puede mejorar.\n"
else:
    resumen_txt += "❌ Sharpe Ratio: Bajo (<1.0). El retorno no compensa el riesgo.\n"

resumen_txt += f"""
Max Drawdown: {max_drawdown:.2%} (máxima caída desde pico)

Próximos pasos sugeridos:
   1. Acumular más datos (3-6 meses) para estadísticas significativas
   2. Si "Alta Confianza" tiene mejor win rate, priorizar esas entradas
   3. Operar máximo 2-3 tickers por semana (capital de $2,000)
   4. Nunca arriesgar más del 5% del capital por trade

═══════════════════════════════════════════════════════════════════
Generado automáticamente desde MTM V6 Backtesting (Google Colab)
"""

# Guardar archivo
ruta_txt = f"{dir_path}/{nombre_base}.txt"
try:
    with open(ruta_txt, 'w', encoding='utf-8') as f:
        f.write(resumen_txt)
    print(f'✅ Archivo guardado exitosamente: {ruta_txt}')
    
    # Verificar que existe
    if os.path.exists(ruta_txt):
        size = os.path.getsize(ruta_txt)
        print(f'   Tamaño: {size} bytes')
        print(f'   Líneas: {len(resumen_txt.splitlines())}')
    else:
        print('⚠️ El archivo no se encontró después de guardarlo')
        
except Exception as e:
    print(f'❌ Error al guardar: {e}')

print('')
print('═══════════════════════════════════════════════════════════════════')
print('📄 PARA GUARDAR EL REPORTE COMPLETO CON GRÁFICOS:')
print('═══════════════════════════════════════════════════════════════════')
print('')
print('Opción 1 — HTML (recomendada):')
print('   1. Menú de Colab: Archivo → Descargar → Descargar .html')
print('   2. Guardalo en: Drive/MTM_V4_Resultados/')
print('   3. Abrilo en Chrome/Firefox (incluye gráficos y tablas)')
print('')
print('Opción 2 — IPYNB (notebook):')
print('   1. Menú de Colab: Archivo → Descargar → Descargar .ipynb')
print('   2. Guardalo en: Drive/MTM_V4_Resultados/')
print('   3. Podés volver a subirlo a Colab y re-ejecutarlo')
print('')
print('Opción 3 — PDF:')
print('   1. Abrí el archivo HTML en Chrome')
print('   2. Ctrl+P → Guardar como PDF')
print('')
print(f'📁 Directorio: {dir_path}')
print(f'🗓️  Próxima ejecución: Viernes de la semana que viene')

---

## ✅ Backtesting completo

Tu resumen ejecutivo está guardado en:
1. **Google Sheet**: hoja `📊 Backtesting Resumen`
2. **Google Drive**: carpeta `MTM_V4_Resultados/Backtesting_V4_YYYYMMDD.txt`

**Recomendación:** Ejecutá este notebook **todos los viernes**.

> 💡 **Tip:** Con menos de 20-30 trades las estadísticas no son significativas. Esperá al menos 2-3 meses de datos antes de hacer cambios drásticos al sistema.